# 00 · Data Sanity Check
Check whether the target can be predicted almost perfectly by a single feature before using this dataset for the classification project.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score

## Load Data

In [2]:
df = pd.read_csv('../data/Student_Depression_Dataset.csv')

## Data Overview

In [3]:
df.head()

,Gender,Age,Family_background,Relationship_status,Marital_status,Living_style,Current_semester,CGPA,friend_circle,Health_condition,Cost_without_study,Home_division,Depression
0,Female,22,Middle class,Single,Unmarried,Bachelor,8,3.43,Yes,Critical,10113,Chittagong,Yes
1,Male,19,Middle class,Single,Unmarried,Bachelor,2,3.70,Yes,Abnormal,13815,Chittagong,Yes
2,Male,23,Middle class,Single,Unmarried,Family,8,3.28,No,Abnormal,15450,Dhaka,Yes
3,Male,20,Middle class,Complicated,Unmarried,Family,2,3.77,No,Good,17413,Dhaka,No
4,Female,21,Middle class,Single,Unmarried,Family,1,4.00,No,Abnormal,16693,Dhaka,Yes


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4464 entries, 0 to 4463
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Gender               4464 non-null   str    
 1   Age                  4464 non-null   int64  
 2   Family_background    4464 non-null   str    
 3   Relationship_status  4464 non-null   str    
 4   Marital_status       4464 non-null   str    
 5   Living_style         4464 non-null   str    
 6   Current_semester     4464 non-null   int64  
 7   CGPA                 4464 non-null   float64
 8   friend_circle        4464 non-null   str    
 9   Health_condition     4464 non-null   str    
 10  Cost_without_study   4464 non-null   int64  
 11  Home_division        4464 non-null   str    
 12  Depression           4464 non-null   str    
dtypes: float64(1), int64(3), str(9)
memory usage: 722.9 KB


In [5]:
print('Duplicated rows:', df.duplicated().sum())

Duplicated rows: 0


In [6]:
df.describe().round(2)

,Age,Current_semester,CGPA,Cost_without_study
count,4464.00,4464.00,4464.00,4464.00
mean,22.62,5.40,3.41,11242.98
std,2.02,2.58,0.39,6640.14
min,17.00,0.00,2.25,0.00
25%,21.00,3.00,3.15,6100.00
50%,23.00,5.00,3.49,9500.00
75%,24.00,7.00,3.70,15439.00
max,28.00,13.00,4.00,32000.00


### Categorical Feature Distributions
Check for rare categories that may cause unstable coefficients in logistic regression.

In [7]:
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    print(df[col].value_counts(), '\n')

Gender
Female    2462
Male      2002
Name: count, dtype: int64 

Family_background
Middle class          2642
Lower Middle class    1151
Rich class             671
Name: count, dtype: int64 

Relationship_status
Single               2803
In a Relationship    1457
Complicated           204
Name: count, dtype: int64 

Marital_status
Unmarried    3619
Married       725
Separated     120
Name: count, dtype: int64 

Living_style
Bachelor    2449
Family      2015
Name: count, dtype: int64 

friend_circle
Yes    3215
No     1249
Name: count, dtype: int64 

Health_condition
Normal      1787
Good        1162
Abnormal     881
Critical     634
Name: count, dtype: int64 

Home_division
Dhaka         1527
Khulna         548
Rajshahi       488
Sylhet         445
Chittagong     400
Rangpur        363
Barisal        355
Mymensingh     338
Name: count, dtype: int64 

Depression
No     3263
Yes    1201
Name: count, dtype: int64 



/var/folders/dc/03c0xx517lscty57znvb7_8m0000gn/T/ipykernel_93542/2543146722.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns


## Target Encoding
Encode `Depression` as a binary target: 1 = Yes, 0 = No.

In [8]:
df['is_depressed'] = (df['Depression'] == 'Yes').astype(int)
df['is_depressed'].value_counts()

is_depressed
0    3263
1    1201
Name: count, dtype: int64

## Class Balance

In [12]:
df['is_depressed'].value_counts(normalize=True).round(4)

is_depressed
0    0.731
1    0.269
Name: proportion, dtype: float64

## Single-Feature AUC
Fit a logistic regression on each feature alone. A single feature with AUC near 1 would indicate the label is almost determined by that feature.

In [10]:
y = df['is_depressed']
features = [c for c in df.columns if c not in ['Depression', 'is_depressed']]

rows = []
for col in features:
    X = pd.get_dummies(df[[col]], drop_first=True, dtype=float)
    X = sm.add_constant(X)
    try:
        res = sm.GLM(y, X, family=sm.families.Binomial()).fit()
        auc = roc_auc_score(y, res.predict(X))
    except Exception:
        auc = np.nan
    rows.append({'feature': col, 'single_var_auc': round(auc, 4)})

pd.DataFrame(rows).sort_values('single_var_auc', ascending=False)

,feature,single_var_auc
9,Health_condition,0.9490
3,Relationship_status,0.6447
4,Marital_status,0.5671
7,CGPA,0.5603
11,Home_division,0.5555
8,friend_circle,0.5524
2,Family_background,0.5469
1,Age,0.5250
10,Cost_without_study,0.5225
5,Living_style,0.5200


## Depression Rate by Health Condition

In [11]:
pd.crosstab(df['Health_condition'], df['Depression'], normalize='index').round(3)

Depression,No,Yes
Health_condition,,
Abnormal,0.451,0.549
Critical,0.000,1.000
Good,1.000,0.000
Normal,0.954,0.046


In [14]:
cat_features = df.select_dtypes(include='str').columns.drop('Depression')
for col in cat_features:
    print(pd.crosstab(df[col], df['Depression'], normalize='index').round(3), '\n')

Depression     No    Yes
Gender                  
Female      0.734  0.266
Male        0.727  0.273 

Depression             No    Yes
Family_background               
Lower Middle class  0.694  0.306
Middle class        0.723  0.277
Rich class          0.824  0.176 

Depression              No    Yes
Relationship_status              
Complicated          0.868  0.132
In a Relationship    0.885  0.115
Single               0.641  0.359 

Depression         No    Yes
Marital_status              
Married         0.880  0.120
Separated       0.800  0.200
Unmarried       0.699  0.301 

Depression       No    Yes
Living_style              
Bachelor      0.717  0.283
Family        0.748  0.252 

Depression        No    Yes
friend_circle              
No             0.657  0.343
Yes            0.760  0.240 

Depression           No    Yes
Health_condition              
Abnormal          0.451  0.549
Critical          0.000  1.000
Good              1.000  0.000
Normal            0.954  0.046 



In [15]:
y = df['is_depressed']
X = df.drop(columns=['Depression', 'is_depressed', 'Health_condition'])
X = pd.get_dummies(X, drop_first=True, dtype=float)
X = sm.add_constant(X)

model = sm.GLM(y, X, family=sm.families.Binomial())
res = model.fit()
print('AUC without Health_condition (in-sample):', round(roc_auc_score(y, res.predict(X)), 4))

AUC without Health_condition (in-sample): 0.7268


## Conclusion
- `Health_condition` alone achieves an in-sample AUC of 0.949. All students with *Critical* health are labeled depressed and none with *Good* health are, indicating the label is largely rule-based on this feature. It is excluded from modeling to avoid leakage.
- No other categorical feature shows a deterministic pattern (depression rates range from 11% to 36% across categories).
- A logistic regression on all remaining features reaches an in-sample AUC of 0.727, leaving meaningful room for threshold selection and top-k analysis.